# Proyecto Agente de IA: BI Reporter

## Alcance adaptado para bootcamp

La idea original era disenar un agente conectado a Claude y al warehouse real de Basebone mediante MCP. Como todavia no tengo acceso corporativo, el proyecto se adapta a una demo local:

- **Groq + modelo OpenAI-compatible** sustituye a Claude como motor LLM.
- Un **CSV grande en castellano** sustituye al warehouse.
- El agente usa herramientas Python para calcular KPIs.
- El LLM redacta el email final con los resultados de esas herramientas.
- El email real se sustituye por un **borrador local seguro** en TXT y HTML.


## Patron tecnico del agente

El flujo de ejecucion del agente se organiza en cinco pasos:

1. **Percibir:** el agente recibe la peticion del reporte diario.
2. **Razonar:** el LLM decide que herramienta necesita.
3. **Actuar:** Python ejecuta la herramienta pedida.
4. **Observar:** el resultado vuelve al LLM.
5. **Responder:** el LLM redacta el email final.

La diferencia es que nuestras herramientas son de negocio: analisis de KPIs, top canales, top aliases y anomalias.


## Implementacion modular

- `src/config.py`
- `src/carga_datos.py`
- `src/analisis_kpis.py`
- `src/reporte.py`
- `src/email_reporter.py`
- `src/llm_client.py`
- `src/herramientas_agente.py`
- `main.py`


In [ ]:
from src.analisis_kpis import analizar_kpis_y_anomalias
from src.carga_datos import cargar_csv_kpis
from src.config import EMAIL_DESTINO, EMAIL_REMITENTE, FECHA_OBJETIVO_DEMO, OUTPUTS_DIR
from src.email_reporter import guardar_borrador_email, preparar_email_reporte
from src.herramientas_agente import HerramientasBI
from src.reporte import generar_reporte_texto


## Paso 1: cargar y validar el CSV


In [ ]:
df_kpis, resumen_csv = cargar_csv_kpis()
resumen_csv


In [ ]:
df_kpis.head()


## Paso 2: analizar KPIs y anomalias con pandas


In [ ]:
resultados_analisis = analizar_kpis_y_anomalias(
    df_kpis,
    fecha_objetivo=FECHA_OBJETIVO_DEMO,
    ventana_historica_dias=14,
    umbral_variacion=0.20,
)

resultados_analisis["comparacion_global"]


In [ ]:
resultados_analisis["anomalias_dimensiones"].head(20)


## Paso 3: probar las herramientas del agente

Estas son las funciones que el LLM podra pedir durante el bucle agente.


In [ ]:
herramientas_bi = HerramientasBI(destinatario=EMAIL_DESTINO)
print(herramientas_bi.analizar_dia_bi({"fecha": FECHA_OBJETIVO_DEMO})[:1500])


In [ ]:
print(herramientas_bi.obtener_top_canales({"fecha": FECHA_OBJETIVO_DEMO, "n": 5}))


## Paso 4: ejecutar el agente LLM con Groq

Esta celda pide tu API key y la guarda solo en la variable de entorno de esta sesion.
No se escribe en ningun archivo.


In [ ]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Pega aqui tu API key de Groq: ")


In [ ]:
from src.agente_bi_function_calling import ejecutar_agente_bi_function_calling

resultado_agente = ejecutar_agente_bi_function_calling(
    fecha=FECHA_OBJETIVO_DEMO,
    destinatario=EMAIL_DESTINO,
    verbose=True,
)

print(resultado_agente["respuesta"])


## Paso 5: guardar el borrador de email generado por el agente


In [ ]:
email = preparar_email_reporte(
    resultados={"fecha_objetivo": FECHA_OBJETIVO_DEMO},
    reporte_markdown=resultado_agente["respuesta"],
    destinatario=EMAIL_DESTINO,
    remitente=EMAIL_REMITENTE,
)

rutas_email = guardar_borrador_email(email, OUTPUTS_DIR)
rutas_email


## Ejecucion completa desde terminal

BI Reporter usa siempre el LLM con function calling. No existe modo `--sin-llm`.

```powershell
$env:GROQ_API_KEY="tu_api_key"
python main.py --fecha 2026-06-29
```

El flag `--usar-llm` se conserva por compatibilidad, pero ya no es necesario porque el agente LLM se ejecuta siempre.


## Envio real por Gmail con Composio

El proyecto ya no usa SMTP ni app passwords. Gmail se conecta mediante Composio y OAuth.

```powershell
$env:COMPOSIO_API_KEY="comp_..."
$env:COMPOSIO_USER_ID="tu_usuario_composio"
python conectar_gmail_composio.py
```

Despues de autorizar Gmail en el navegador:

```powershell
$env:GROQ_API_KEY="tu_api_key"
$env:COMPOSIO_API_KEY="comp_..."
$env:COMPOSIO_USER_ID="tu_usuario_composio"
python main.py --enviar-email
```

Las claves no se pasan al modelo: el LLM pide la herramienta y Composio ejecuta Gmail con OAuth.


## Function calling nativo aplicado al proyecto

La version actual usa function calling nativo: el modelo recibe `tools`, pide `tool_calls`, Python ejecuta la funcion y devuelve un mensaje con rol `tool`.


In [ ]:
from src.agente_bi_function_calling import TOOLS_BI

[tool['function']['name'] for tool in TOOLS_BI]


## Credenciales y function calling

Las contrasenas no se configuran mediante function calling. El modelo puede pedir una herramienta, pero los secretos viven en variables de entorno (`GROQ_API_KEY`, `COMPOSIO_API_KEY`) y Python los usa internamente sin mostrarselos al LLM.


## Composio como plataforma de herramientas

Composio permite usar herramientas reales como Gmail sin programar SMTP ni gestionar app passwords. El agente sigue usando function calling, pero Composio proporciona y ejecuta la herramienta de Gmail.
